In [3]:
import sqlite3
conn = sqlite3.connect("../outputs/hpo/fixed_hpo/optuna.db")

In [4]:
import json
from src.hpo.optuna_trial import compute_trial_grouped_z_score
from pathlib import Path

with open("./baseline_metrics.json", "r") as f:
    baseline_metrics = json.load(f)

with open("../outputs/hpo/fixed_hpo/outputs.json", "r") as f:
    hpo_data = json.load(f)

In [5]:
trial_scores = {}
for trial in hpo_data.keys():
    trial_scores[trial] = {}

for dataset_name in ("nt_benchmark", "genomic_benchmark", "bend_benchmark", "chromatin_profile_prediction"):
    def get_tasks(trial):
        global dataset_name
        if dataset_name == "chromatin_profile_prediction":
            cpp = trial.get("chromatin_profile_prediction")
            if cpp is not None:
                return {"chromatin_profile_prediction": cpp}
            else:
                return None
        else:
            return trial.get(dataset_name)

    for trial in hpo_data.keys():
        tasks = get_tasks(hpo_data[trial])

        if tasks is not None:
            for name, val in tasks.items():
                if name == "pretraining": 
                    continue
                trial_scores[trial][name] = val[0][baseline_metrics[name]["metric"]]

In [6]:
from copy import deepcopy
z_scores = deepcopy(trial_scores)

# Turn trial_scores into z_scores
for trial in trial_scores:
    for task in trial_scores[trial]:
        dataset_metrics = baseline_metrics[task]
        score = trial_scores[trial][task]
        score = (score - dataset_metrics["mean"]) / dataset_metrics["std"]
        z_scores[trial][task] = score

In [8]:
scores = []
for trial in z_scores:
    mean = 0
    for task in z_scores[trial]:
        mean += z_scores[trial][task]
    mean = mean / len(z_scores[trial].keys())
    scores.append(mean)

In [9]:
print(scores)

[-3.244741199951053, -2.0970295938008827, -1.8641625964197008, -4.612179108729117, -1.3387500571332147, -2.867867033914606, -0.5411336701397793, -0.5753241159014064, -1.8953700892503123, -1.631531067216069, -3.969361507397927, -3.818050544403167, -1.7111522242977497, -5.1775808232978715, -3.8315203342703317, -5.449259071486706, -4.11534780695314, -1.209954356668473, -1.9222352934412383, -2.7356730126541753, -0.03344321553814691]


In [10]:
cursor = conn.cursor()

for trial, score in enumerate(scores):
    cursor.execute(f"UPDATE trial_values SET value={score} WHERE trial_id={trial + 1}")

In [11]:
conn.commit()
conn.close()